# Capstone: Fool a Real Classifier, Then Chart Its Robustness

Capstone notebook for the [Adversarial Examples](https://josephausterweil.github.io/probintro/ethics/adversarial-examples/) chapter (Part IX).

**The experiment.** Three acts on real handwritten-digit images:

- **Part A — implement FGSM.** We provide a trained linear classifier on a *confusable* digit pair (**3 vs 5**). You write the **fast gradient sign method** (Goodfellow, Shlens & Szegedy, 2015): the gradient of the loss with respect to the *input*, stepped along its sign. Then visualize one attack three ways — original, perturbation, adversarial.
- **Part B — the robustness curve.** Sweep the perturbation budget `epsilon`, batch-attack the whole test set at each level, and plot **accuracy vs. perturbation size**.
- **Part C — stretch: adversarial training.** Fold the adversarial examples back into training, retrain, and redraw the curve. Confirm the chapter's Tanaka-san box quantitatively: the curve lifts, but does not go flat.

Setup, data, and the trained classifier are **complete** — run them as-is. The attack, the sweep, and the defense are **TODO** stubs; each says exactly what to build, with a commented check for what success looks like.

**Anchors.** Goodfellow, Shlens & Szegedy (2015), *Explaining and harnessing adversarial examples*, introduced FGSM and the "models are too linear in high dimensions" explanation you are reproducing. Szegedy et al. (2014), *Intriguing properties of neural networks*, first reported that an imperceptible perturbation could flip a state-of-the-art classifier — the discovery this whole chapter unpacks.

## 0. Setup (complete — run me)

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

key = jr.PRNGKey(0)
print('JAX ready:', jax.__version__)

## Part A — implement FGSM

### A1. A trained classifier to attack (complete — run me)

8x8 grayscale handwritten digits, pixels rescaled to 0-1. We keep the **confusable pair 3 vs 5** — unlike 0 vs 1, these are genuinely hard, so the model ends up confident but *not* saturated, which is the interesting regime for an attack (a saturated model has near-zero input gradients and is boringly hard to nudge). We split off a test set and train a linear classifier: one sigmoid neuron, the dot-product-plus-threshold that is exactly Goodfellow's "too linear" model.

Everything the attack needs is defined here: `prob(x)` (the kiosk's probability that image `x` is a **5**) and the held-out `X_test`, `y_test`.

In [ ]:
digits = load_digits()
mask = (digits.target == 3) | (digits.target == 5)
X_all = jnp.array(digits.data[mask] / 16.0)                    # pixels in [0, 1]
y_all = jnp.array((digits.target[mask] == 5).astype(float))   # 1.0 = digit "5"

perm = jr.permutation(jr.PRNGKey(0), X_all.shape[0])
n_train = int(0.8 * X_all.shape[0])
tr, te = perm[:n_train], perm[n_train:]
X, y = X_all[tr], y_all[tr]
X_test, y_test = X_all[te], y_all[te]

def logit(params, x):
    return jnp.dot(params['w'], x) + params['b']

def bce(params, Xb, yb):
    p = jax.nn.sigmoid(jax.vmap(lambda x: logit(params, x))(Xb))
    e = 1e-7
    return -jnp.mean(yb * jnp.log(p + e) + (1 - yb) * jnp.log(1 - p + e))

params = {'w': jnp.zeros(64), 'b': jnp.array(0.0)}
grad_w = jax.grad(lambda pr: bce(pr, X, y))      # gradient w.r.t. WEIGHTS: this is training
for step in range(400):
    params = jax.tree.map(lambda p, g: p - 0.3 * g, params, grad_w(params))

def prob(x):                     # the kiosk's probability that image x is a "5"
    return jax.nn.sigmoid(logit(params, x))

test_acc = float(jnp.mean((jax.vmap(prob)(X_test) > 0.5) == (y_test > 0.5)))
print(f'test accuracy on 3-vs-5: {test_acc:.3f}')      # ~0.97 -- a good classifier

### A2. The attack — TODO

Write the fast gradient sign method. For an image `x` with true label `t`, the attacker wants the direction in pixel-space that *raises* the loss of the true label, then steps a tiny `epsilon` along its **sign**:

$$x_{adv} = \mathrm{clip}\big(x + \epsilon \cdot \mathrm{sign}(\nabla_x L),\; 0,\, 1\big).$$

The clip keeps pixels valid (0-1). The gradient is `jax.grad` of a per-example loss taken **with respect to `x`** — the exact mirror of the `grad_w` that trained the model.

In [ ]:
def fgsm(x, t, epsilon):
    """Return the adversarial image x_adv for one example (image x, label t)."""
    # TODO:
    #   1. def input_loss(xx):
    #          p = prob(xx); e = 1e-7
    #          return -(t * jnp.log(p + e) + (1 - t) * jnp.log(1 - p + e))
    #   2. grad_x = jax.grad(input_loss)(x)              # gradient w.r.t. the INPUT
    #   3. return jnp.clip(x + epsilon * jnp.sign(grad_x), 0.0, 1.0)
    raise NotImplementedError

# When fgsm works, this should flip a confident, correctly-classified "5":
#
# probs = jax.vmap(prob)(X_test)
# correct5 = (y_test > 0.5) & (probs > 0.5)
# i = int(jnp.argmin(jnp.where(correct5, jnp.abs(probs - 0.9), 1e9)))  # a confident (~0.94) correct "5"
# x0, t0 = X_test[i], y_test[i]
# x_adv = fgsm(x0, t0, 0.2)
# print(f'clean: prob(5)={float(prob(x0)):.3f}   adversarial: prob(5)={float(prob(x_adv)):.3f}')
# (expect clean ~0.94 and adversarial ~0.23 -- below 0.5, so the confident 5 now reads as a 3.
#  Note: pick a confident-but-not-saturated point; the MOST confident 5 has a near-zero
#  input gradient and resists -- a real, teachable FGSM caveat.)

### A3. See the whisper — TODO

Once `fgsm` works, uncomment the block below (it is complete) to view one attack three ways: the **original** digit, the **perturbation** on its own (this is the "sticker"), and the **adversarial** digit the classifier now gets wrong. The middle panel is the whole point — squint, and the left and right images look identical.

In [ ]:
# probs = jax.vmap(prob)(X_test)
# correct5 = (y_test > 0.5) & (probs > 0.5)
# i = int(jnp.argmin(jnp.where(correct5, jnp.abs(probs - 0.9), 1e9)))
# x0, t0 = X_test[i], y_test[i]
# x_adv = fgsm(x0, t0, 0.2)
#
# fig, ax = plt.subplots(1, 3, figsize=(7, 2.6))
# ax[0].imshow(x0.reshape(8, 8), cmap='gray_r')
# ax[0].set_title(f'original\nprob(5)={float(prob(x0)):.2f}', fontsize=9)
# ax[1].imshow((x_adv - x0).reshape(8, 8), cmap='bwr')          # the perturbation itself
# ax[1].set_title('perturbation\neps * sign(grad)', fontsize=9)
# ax[2].imshow(x_adv.reshape(8, 8), cmap='gray_r')
# ax[2].set_title(f'adversarial\nprob(5)={float(prob(x_adv)):.2f}', fontsize=9)
# for a in ax: a.axis('off')
# plt.tight_layout(); plt.show()
pass

## Part B — the robustness curve — TODO

The chapter's little `epsilon` column, full size. Attack **every test image** at each perturbation budget and record the classifier's accuracy on the attacked set. Plot accuracy (y) vs. `epsilon` (x): a plateau near the clean accuracy that collapses once `epsilon` clears the (tiny) distance to the boundary.

In [ ]:
# TODO:
#   epsilons = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3]
#   accs = []
#   for eps in epsilons:
#       X_adv = jax.vmap(lambda x, t: fgsm(x, t, eps))(X_test, y_test)   # batch attack
#       acc = float(jnp.mean((jax.vmap(prob)(X_adv) > 0.5) == (y_test > 0.5)))
#       accs.append(acc)
#   plt.plot(epsilons, accs, 'o-'); plt.xlabel('perturbation budget  epsilon')
#   plt.ylabel('test accuracy'); plt.ylim(0, 1); plt.grid(alpha=0.3)
#   plt.title('robustness curve: 3 vs 5'); plt.show()
#
# (expect ~0.97 at eps=0 falling toward ~0 by eps~0.3 -- an easily fooled model.)
pass

## Part C — stretch: adversarial training

Tanaka-san's "just train on the stickered ones" as an experiment. At each training step, craft FGSM versions of the batch (at a fixed `epsilon`, say 0.2) and train on the **union** of clean and adversarial images with their *correct* labels. Then redraw the Part-B curve for the hardened model on top of the original.

Confirm the chapter's claim: the curve **lifts** (the model resists the `epsilon` it was trained against) but does **not** go flat — push `epsilon` higher, or use a multi-step attack, and accuracy still falls. Defense raises the bar; it does not end the arms race.

In [ ]:
# TODO (stretch):
#   1. Make fgsm take a params argument so it attacks the CURRENT model:
#        def fgsm_p(params, x, t, eps): ... prob_p(params, x) ... (mirror A2)
#   2. params_adv = {'w': jnp.zeros(64), 'b': jnp.array(0.0)}
#   3. Training loop: each step build X_adv = vmap(fgsm_p)(params_adv, X, y, 0.2),
#      then step bce on the stack (jnp.concatenate([X, X_adv]), jnp.concatenate([y, y])).
#   4. Recompute the Part-B curve for params_adv; plot both curves together.
#   5. Read off: how much did eps=0.2 accuracy improve? Is the curve flat at eps=0.3?
pass

---

## References

- Szegedy, C., Zaremba, W., Sutskever, I., Bruna, J., Erhan, D., Goodfellow, I., & Fergus, R. (2014). Intriguing properties of neural networks. *ICLR*. <https://arxiv.org/abs/1312.6199>
- Goodfellow, I. J., Shlens, J., & Szegedy, C. (2015). Explaining and harnessing adversarial examples. *ICLR*. <https://arxiv.org/abs/1412.6572>

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*